# Distance to Coast — Phase 2, Notebook 2

Computes `dist_to_coast_km` for every U.S. bridge: the great-circle distance to
the nearest **ocean** coastline. This captures airborne marine-salt exposure —
the single biggest corrosion signal the freeze/snow MA feature set omits.

**Method:** Natural Earth `ne_10m_coastline` (ocean coasts only; Great Lakes are
freshwater and live in a separate lakes layer, so they are correctly excluded).
A `sklearn` `BallTree` with the **haversine** metric gives true great-circle
nearest-distance and works globally in one pass, so AK/HI need no special CRS.

**Output:** `Bridges_all_of_US/us_bridge_coastal_distance.csv`
(keys: STRUCTURE_NUMBER_008, STATE_FIPS, dist_to_coast_km). Static per bridge.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import geopandas as gpd
from sklearn.neighbors import BallTree
import urllib.request

print("geopandas", gpd.__version__)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
RAW_DIR   = Path("raw")
GEO_CACHE = Path("geo_cache"); GEO_CACHE.mkdir(exist_ok=True)
COAST_ZIP = GEO_CACHE / "ne_10m_coastline.zip"
OUT       = Path("us_bridge_coastal_distance.csv")
EARTH_R_KM = 6371.0

COAST_URLS = [
    "https://naciscdn.org/naturalearth/10m/physical/ne_10m_coastline.zip",
    "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip",
]
print("config ready")

In [ ]:
# ── Load per-bridge coordinates: valid-first + county-centroid imputation ─────
# (matches download_climate_national.ipynb so cell_map & coastal cover the same bridges)
def nbi_to_decimal(val):
    degrees = val // 1000000
    minutes = (val % 1000000) // 10000
    seconds = (val % 10000) / 100
    return degrees + minutes / 60 + seconds / 3600

# Memory-safe: stream each per-state file in 200K-row chunks. VALID-COORD-FIRST: filter to
# rows with a good in-bounds coordinate BEFORE deduping, so a bridge whose first survey row
# has a bad coord is still captured from a later good row (the old dedup-then-filter dropped
# ~130K such bridges). Then COUNTY-CENTROID IMPUTE the ~8% with NO coord in any year.
valid_parts, all_parts = [], []
for f in sorted(RAW_DIR.glob("*_bridges.csv")):
    vp, ap = [], []
    for chunk in pd.read_csv(
            f, usecols=["STRUCTURE_NUMBER_008", "STATE_FIPS", "COUNTY_CODE_003", "LAT_016", "LONG_017"],
            dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str, "COUNTY_CODE_003": str}, chunksize=200_000):
        st = chunk["STATE_FIPS"].str.strip(); sn = chunk["STRUCTURE_NUMBER_008"].str.strip()
        co = chunk["COUNTY_CODE_003"].str.strip()
        lat_raw = pd.to_numeric(chunk["LAT_016"], errors="coerce")
        lon_raw = pd.to_numeric(chunk["LONG_017"], errors="coerce")
        lat = nbi_to_decimal(lat_raw); lon = -nbi_to_decimal(lon_raw)   # West = negative
        v = lat_raw.gt(0) & lon_raw.gt(0) & lat.between(15, 72) & lon.between(-170, -60)
        ap.append(pd.DataFrame({"STATE_FIPS": st, "STRUCTURE_NUMBER_008": sn, "COUNTY_CODE_003": co}))
        vp.append(pd.DataFrame({"STATE_FIPS": st[v], "STRUCTURE_NUMBER_008": sn[v],
                                "COUNTY_CODE_003": co[v], "lat": lat[v], "lon": lon[v]}))
    valid_parts.append(pd.concat(vp, ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"]))
    all_parts.append(pd.concat(ap, ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"]))
valb = pd.concat(valid_parts, ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"])
allb = pd.concat(all_parts,   ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"])

cty = valb.groupby(["STATE_FIPS","COUNTY_CODE_003"])[["lat","lon"]].mean().reset_index()
sta = valb.groupby(["STATE_FIPS"])[["lat","lon"]].mean().reset_index().rename(columns={"lat":"slat","lon":"slon"})
vkeys = set(map(tuple, valb[["STATE_FIPS","STRUCTURE_NUMBER_008"]].values))
cl = allb[~allb.set_index(["STATE_FIPS","STRUCTURE_NUMBER_008"]).index.isin(vkeys)].copy()
cl = cl.merge(cty, on=["STATE_FIPS","COUNTY_CODE_003"], how="left").merge(sta, on="STATE_FIPS", how="left")
cl["lat"] = cl["lat"].fillna(cl["slat"]); cl["lon"] = cl["lon"].fillna(cl["slon"])
cl = cl.dropna(subset=["lat","lon"])

valb["location_imputed"] = np.int8(0); cl["location_imputed"] = np.int8(1)
b = (pd.concat([valb[["STATE_FIPS","STRUCTURE_NUMBER_008","lat","lon","location_imputed"]],
                cl[["STATE_FIPS","STRUCTURE_NUMBER_008","lat","lon","location_imputed"]]], ignore_index=True)
       .drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"]).reset_index(drop=True))
print(f"{len(b):,} bridges ({int(b['location_imputed'].sum()):,} county-imputed) with coordinates")

In [ ]:
# ── Download + load the Natural Earth ocean coastline ────────────────────────
if not COAST_ZIP.exists():
    for url in COAST_URLS:
        try:
            print("downloading", url)
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (research)"})
            with urllib.request.urlopen(req, timeout=120) as r, open(COAST_ZIP, "wb") as out:
                out.write(r.read())
            print("saved", COAST_ZIP, COAST_ZIP.stat().st_size, "bytes")
            break
        except Exception as e:
            print("  failed:", e)
coast = gpd.read_file(COAST_ZIP)
print(coast.shape, "coastline features; CRS", coast.crs)

In [ ]:
# ── Build coastline vertex array and a haversine BallTree ────────────────────
pts = []
for geom in coast.geometry:
    if geom is None:
        continue
    lines = [geom] if geom.geom_type == "LineString" else list(geom.geoms)
    for ln in lines:
        pts.append(np.asarray(ln.coords)[:, :2])   # (n, 2) as lon, lat
coast_xy = np.vstack(pts)
coast_rad = np.radians(coast_xy[:, [1, 0]])        # reorder to (lat, lon) for haversine
tree = BallTree(coast_rad, metric="haversine")
print(f"{len(coast_rad):,} coastline vertices indexed")

q = np.radians(b[["lat", "lon"]].values)
dist, _ = tree.query(q, k=1)
b["dist_to_coast_km"] = dist[:, 0] * EARTH_R_KM
print("dist_to_coast_km computed")

In [ ]:
# ── Save + sanity checks ─────────────────────────────────────────────────────
out = b[["STRUCTURE_NUMBER_008", "STATE_FIPS", "dist_to_coast_km"]].copy()
out.to_csv(OUT, index=False)
print(f"{len(out):,} rows -> {OUT}")
print(out["dist_to_coast_km"].describe().round(1).to_string())

# Regional spot checks: coastal states should be near 0; interior should be large.
fips_name = {"25": "MA(coastal)", "12": "FL(coastal)", "06": "CA(coastal)",
             "20": "KS(interior)", "30": "MT(interior)", "46": "SD(interior)"}
chk = (out.assign(state=out.STATE_FIPS.map(fips_name)).dropna(subset=["state"])
          .groupby("state")["dist_to_coast_km"].median().round(1))
print("\nMedian dist_to_coast_km by state:")
print(chk.to_string())